In [33]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [34]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [35]:
data = pd.read_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/ham_data.csv')

In [36]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [38]:
data[data["yesil_skor_notu"].isna()].sample(10)

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
20631,https://world.openfoodfacts.org/product/858400...,8.584002e+12,chocolate pudding – Dr. Oetker – 46g,46.0,NaN,dr. oetker,"Desserts, ,, Puddings",NaN,NaN,NaN,...,2.0,0.0,0.0,1.0,0.0,0.0,chocolate pudding,g,"[Desserts, Puddings]",[]
19776,https://world.openfoodfacts.org/product/325901...,3.259011e+12,Macaronis Mais et Sarrasin sans gluten – Valpi...,500.0,Cardboard,valpibio,"Plant-based foods and beverages, ,, Plant-base...","No gluten, ,, Organic, ,, EU Organic, ,, IT-BI...",NaN,Italie,...,0.0,0.0,0.0,2.0,0.0,0.0,Macaronis Mais et Sarrasin sans gluten,g,"[Plant-based foods and beverages, Plant-based ...","[No gluten, Organic, EU Organic, IT-BIO-009, M..."
48681,https://world.openfoodfacts.org/product/019396...,1.939680e+11,Lightly Salted Deluxe Mixed Nuts – Member's Ma...,34.0,NaN,member's mark,"Plant-based foods and beverages, ,, Plant-base...",NaN,NaN,NaN,...,2.0,5.0,1.0,NaN,4.0,0.0,Lightly Salted Deluxe Mixed Nuts,oz,"[Plant-based foods and beverages, Plant-based ...",[]
36695,https://world.openfoodfacts.org/product/292860...,2.928604e+07,Calabrian Mushroom & Truffle Soup – Marks & Sp...,560.0,NaN,marks & spencer,"Meals, ,, Soups, ,, Reheatable soups",Green Dot,Italy,NaN,...,0.0,4.0,3.0,0.0,0.0,0.0,Calabrian Mushroom & Truffle Soup,g,"[Meals, Soups, Reheatable soups]",[Green Dot]
56447,https://world.openfoodfacts.org/product/003400...,3.400048e+10,Reese's Cup White Cream King Size – Reeses – 2...,2.8,NaN,reeses,"Snacks, ,, Sweet snacks, ,, Frozen foods, ,, Bars",À Lhuile De Tournesol,NaN,NaN,...,13.0,10.0,4.0,NaN,0.0,0.0,Reese's Cup White Cream King Size,oz,"[Snacks, Sweet snacks, Frozen foods, Bars]",[À Lhuile De Tournesol]
4347,https://world.openfoodfacts.org/product/356007...,3.560072e+12,Végétal Émincés Stukjes Saveur Curry Smaak Cur...,160.0,NaN,sensation,"Meat alternatives, ,, fr:A, ,, fr:Vegetal-eminces","Vegetarian, ,, Source of proteins, ,, Vegan, ,...",NaN,NaN,...,0.0,0.0,3.0,7.0,0.0,0.0,Végétal Émincés Stukjes Saveur Curry Smaak Curry,g,"[Meat alternatives, fr:A, fr:Vegetal-eminces]","[Vegetarian, Source of proteins, Vegan, High p..."
22200,https://world.openfoodfacts.org/product/520102...,5.201024e+12,Πατατάκια Μπάρμπεκιου,NaN,NaN,NaN,el:Γαριδακια,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,0.0,0.0,Πατατάκια Μπάρμπεκιου,None,[el:Γαριδακια],[]
39544,https://world.openfoodfacts.org/product/318123...,3.181232e+12,Happy Family x3 – Charal – 300 g (3 x 100 g),300.0,"Box, ,, Cardboard, ,, Frozen, ,, Tray",charal,"Meats and their products, ,, Beef and its prod...","French meat, ,, French beef, ,, French poultry...",France,France,...,0.0,6.0,3.0,NaN,0.0,0.0,Happy Family x3,g,"[Meats and their products, Beef and its produc...","[French meat, French beef, French poultry, Nut..."
5157,https://world.openfoodfacts.org/product/619157...,6.191578e+12,Eau – Dima,NaN,NaN,dima,"Beverages and beverages preparations, ,, Bever...",NaN,Tunisia,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Eau,None,"[Beverages and beverages preparations, Beverag...",[]
30718,https://world.openfoodfacts.org/product/900690...,9.006900e+12,Almdudler zuckerfrei – 1l,1.0,NaN,almdudler,"Beverages and beverages preparations, ,, Bever...","Low or no sugar, ,, Vegetarian, ,, Vegan, ,, N...",Austria,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Almdudler zuckerfrei,l,"[Beverages and beverages preparations, Beverag...","[Low or no sugar, Vegetarian, Vegan, No sugar]"


In [39]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [40]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001580
urun_adi                         0.003160
miktar                          18.674546
ambalaj                         57.499526
markalar                         3.435308
kategoriler                      0.003160
etiketler                       30.001264
mensei                          75.346059
uretim_yerleri                  79.520890
satildigi_ulkeler                0.104292
icerik_metni                    12.089944
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   12.413880
nutriscore_notu                  0.093231
nova_grubu                      14.379622
yesil_skor_notu                 25.695278
palmiye_yagi_icermez            18.247898
vejetaryen                      22.125656
vegan_durumu                    11.912964
yag_seviyesi                     2.156943
doymus_yag_seviyesi              3.150875
seker_seviyesi                   2

In [41]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)
# ,"etiketler"

In [49]:
data["etiketler"].head(20)

0                                            Vegetarian
1                                                   NaN
2     ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...
3                                             Green Dot
4                                                Triman
5                                                   NaN
6                                                   NaN
7                                                   NaN
8                                   Green Dot, ,, Maroc
9                                                   NaN
10                                            Green Dot
11    French milk, ,, Made in France, ,, Nutriscore,...
12    Fair trade, ,, Source of fibre, ,, High fibres...
13    Vegetarian, ,, Fair trade, ,, No gluten, ,, Or...
14    ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...
15    Sustainable, ,, No preservatives, ,, Source of...
16    No gluten, ,, No preservatives, ,, FSC, ,, Gre...
17                                              

In [43]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [44]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (63092): ['1.020304050607081e+17', '1000029852900.0', '10001219.0', '10001295.0', '10001356.0', '10001400.0', '10001404.0', '10001691.0', '10001707.0', '10001875.0'] ...
miktar (1133): ['0.0', '0.03', '0.042', '0.045', '0.046', '0.05', '0.055', '0.06', '0.065', '0.07'] ...
markalar (12268): ['"tradition culinaire"', "'z bregov", '(sans marque)', '07x netto 03.25', '1 2 3 fruits', '1 attimo in forma', '1 l', '1 x auer 01.25', '1%', '1-2-3'] ...
etiketler (18879): ['1', '1 of 5 a day', '1% for the planet', '1% for the planet, ,, Triman', '10 Percent Coconut Water BCAA 700mg Eleytrocltyes Antioxidant B Vits', '100% Milch aus Österreich', '100% arabica', '100% arabica, ,, Triman', '100% arabica, ,, pt:Certificado Gourmet ABIC', '100% italian'] ...
alerjenler (1800): ["['Acesulfame-potassium']", "['Apple', 'Banana', 'Celery', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Nuts']", "['Apple', 'Banana', 'Gluten', 'Sulphur 

In [45]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     39049
False    12687
Name: count, dtype: int64

In [46]:
data.isnull().mean() * 100

barkod                         0.001580
miktar                        18.674546
markalar                       3.435308
etiketler                     30.001264
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 12.413880
nutriscore_notu                0.093231
nova_grubu                    14.379622
yesil_skor_notu               25.695278
palmiye_yagi_icermez          18.247898
vejetaryen                    22.125656
vegan_durumu                  11.912964
enerji_kcal                    0.967069
yag_g                          0.973390
doymus_yag_g                   2.009987
karbonhidrat_g                 1.053979
seker_g                        1.426901
lif_g                         29.302825
protein_g                      0.979711
tuz_g                          0.774287
alkol_yuzde                   95.049302
meyve_sebze_baklagil_yuzde    72.090892
birim                         20.629227
kategori_listesi               0.000000


In [47]:
data.to_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/data.csv', index=False)

In [48]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'etiketler', 'alerjenler',
       'eser_miktarlar', 'icerik_sayisi', 'nutriscore_notu', 'nova_grubu',
       'yesil_skor_notu', 'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu',
       'enerji_kcal', 'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g',
       'lif_g', 'protein_g', 'tuz_g', 'alkol_yuzde',
       'meyve_sebze_baklagil_yuzde', 'birim', 'kategori_listesi',
       'etiketler_listesi'],
      dtype='object')